# TFBS project: train, compare and explain all models (FOSL2, HepG2)

**What this notebook does, top to bottom**
1. Loads the real dataset: ENCODE ChIP-seq for **FOSL2** in HepG2 liver cells (101 bp, ~39k training and ~10k test sequences, from the public BERT-TFBS benchmark). It's the same data the original CNN was trained on.
2. Downloads the **JASPAR** database of known binding motifs (used for the PWM baseline and to name the motifs the model learns).
3. Trains **8 methods on exactly the same data**, each **5 times** (5-fold cross-validation):
   - JASPAR PWM scan (classic, no training) and k-mer logistic regression (classic machine learning)
   - CNN, CNN + BiLSTM, and the original CNN + BiLSTM + Transformer (v1)
   - **CNN + BiLSTM + Transformer v2 (our model, deeper convolution trunk)**, plus two ablations: v2 without the Transformer and v2 without the BiLSTM
   - All deep models use the same recipe: early stopping on validation AUROC and reverse-complement (both-strand) training
4. Explains the trained model: real sequence logos from the CNN filters, matched to JASPAR, and a comparison of explanation methods (ShiftSmooth vs gradient vs Integrated Gradients).
5. Packs everything into **`My Drive/TFBS/TFBS_results_v2.zip`**. Send that file back to Claude.

**Before you start**
- `Runtime → Change runtime type → T4 GPU`, then Save.
- Upload **`tfbs_project.zip`** (from Claude) to a Google Drive folder called **`TFBS`**, so the path is `My Drive/TFBS/tfbs_project.zip`.
- Then `Runtime → Run all`. Allow Drive access when asked.

**Time:** about 45–90 minutes on the free T4 GPU. Progress is saved to Drive after every finished run. If Colab disconnects, just `Run all` again and it continues where it stopped.


## 1. Check the GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

## 2. Connect Google Drive and unpack the project code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil
WORK = '/content/drive/MyDrive/TFBS'          # your Drive folder
PROJECT = '/content/tfbs_project'             # working copy inside Colab
os.makedirs(WORK, exist_ok=True)

zip_path = os.path.join(WORK, 'tfbs_project.zip')
assert os.path.exists(zip_path), f'Put tfbs_project.zip in {WORK} first.'
shutil.rmtree(PROJECT, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content')
print('Project unpacked:', sorted(os.listdir(PROJECT)))

## 3. Get the dataset (FOSL2, 101 bp)

If you put your own `train.csv` and `test.csv` in `My Drive/TFBS/`, those are used. Otherwise the cell downloads them from the public BERT-TFBS repository.

Each row is `chr_id, sequence, label`. `label = 1`: FOSL2 really binds there (ChIP-seq peak). `label = 0`: a shuffled copy of a real site with the same pairs of letters in a different order (`_shuf`), so the model can't cheat by counting letters.

In [ ]:
import pandas as pd, urllib.request
RAW = f'{PROJECT}/ml/data/raw'
os.makedirs(RAW, exist_ok=True)
BASE = 'https://raw.githubusercontent.com/ZX1998-12/BERT-TFBS/master/Dataset/wgEncodeAwgTfbsHaibHepg2Fosl2V0416101UniPk/'

for name in ('train.csv', 'test.csv'):
    mine = os.path.join(WORK, name)
    if os.path.exists(mine):
        shutil.copy(mine, f'{RAW}/{name}'); print(f'Using your {name} from Drive')
    else:
        urllib.request.urlretrieve(BASE + name, f'{RAW}/{name}'); print(f'Downloaded {name}')

for name in ('train.csv', 'test.csv'):
    df = pd.read_csv(f'{RAW}/{name}')
    print(f"{name}: {len(df)} rows | bound={int((df.label==1).sum())} not bound={int((df.label==0).sum())} | "
          f"lengths {df.sequence.str.len().min()}-{df.sequence.str.len().max()} bp")
    assert set(df.columns) >= {'chr_id', 'sequence', 'label'}, 'unexpected columns'

## 4. Clean the data (the team's existing `prepare_dataset.py`)

In [ ]:
%cd {PROJECT}/ml
!python prepare_dataset.py | tail -n 25

## 5. Download the JASPAR motif database (about 5 minutes, only once)

~900 known human/vertebrate binding motifs. Saved to Drive, so it's downloaded only once.

In [ ]:
import sys
JASPAR_DRIVE = f'{WORK}/jaspar_core_vertebrates.json'
LOCAL = f'{PROJECT}/ml/motifs/jaspar_core_vertebrates.json'
os.makedirs(os.path.dirname(LOCAL), exist_ok=True)
if os.path.exists(JASPAR_DRIVE):
    shutil.copy(JASPAR_DRIVE, LOCAL); print('Using JASPAR file saved on Drive')
else:
    sys.path.insert(0, f'{PROJECT}/ml')
    import motifs
    motifs.fetch_jaspar_core()
    shutil.copy(LOCAL, JASPAR_DRIVE); print('Downloaded and saved to Drive')
import json; db = json.load(open(LOCAL))['motifs']
print(len(db), 'motifs;', 'FOSL2 entries:', [m['id'] for m in db if m['name'].upper() == 'FOSL2'])

## 6. Quick test (about 3 minutes)

Every method for 1 round on a small slice, just to prove everything works before the long run. **The numbers here don't mean anything.** Only check that it ends with a table and no red error. If it errors, send Claude a screenshot of the error.

In [ ]:
# Output shows live below. If anything fails, 'Run all' stops here (before the long run).
!TORCH_NUM_THREADS=4 python train_all.py --quick --out "{WORK}/results_v2"
assert _exit_code == 0, 'Quick training test failed: send Claude a screenshot of the red error above.'
!TORCH_NUM_THREADS=4 python interpret.py --model "{WORK}/results_v2/quick_test/final_models/tri_branch_v2.pth" --out "{WORK}/results_v2/quick_test/interpretation" --n-seqs 300 --n-attr 40
assert _exit_code == 0, 'Quick interpretation test failed: send Claude a screenshot of the red error above.'
print('Quick test passed.')

## 7. Full training (about 45–90 minutes)

8 methods × 5 folds. Keep this tab open. If Colab disconnects, `Run all` again and finished runs are skipped automatically.

In [ ]:
!TORCH_NUM_THREADS=4 python train_all.py --out "{WORK}/results_v2"
assert _exit_code == 0, 'Training stopped with an error: send Claude a screenshot.'

## 8. Results table (for the paper)

Each number is the **average over the 5 runs ± standard deviation**, measured on the untouched test set.
- **AUROC**: how well bound sequences are ranked above unbound ones (1.0 = perfect, 0.5 = guessing). The main number.
- **PR-AUC**: like AUROC, focused on how trustworthy "bound" predictions are.
- **Accuracy / Precision / Recall / F1 / MCC**: at the cut-off chosen on validation data (MCC: −1..1, 0 = guessing).

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{WORK}/results_v2/results_table.md').read()))
display(Image(f'{WORK}/results_v2/roc_curves.png', width=500))

## 9. Explain the trained model (about 10 minutes)

- **Filter logos**: the DNA patterns our model's first layer learned, and which known JASPAR motif each looks like. For FOSL2 we expect the AP-1 site **TGA(C/G)TCA**.
- **Explanation-method comparison**: which method's important positions land on the real FOSL2 site.

In [ ]:
import json
best = json.load(open(f'{WORK}/results_v2/final_models/best_model.json'))
print('Best model on VALIDATION data:', best['name'], '| mean validation AUROC', round(best['mean_validation_auroc'], 4))
BEST_FILE = f"{WORK}/results_v2/final_models/{best['file']}"
!TORCH_NUM_THREADS=4 python interpret.py --model "{BEST_FILE}" --out "{WORK}/results_v2/interpretation"
assert _exit_code == 0, 'Interpretation stopped with an error: send Claude a screenshot.'
from IPython.display import Markdown, Image, display
display(Image(f'{WORK}/results_v2/interpretation/filter_logos.png', width=900))
display(Markdown(open(f'{WORK}/results_v2/interpretation/attribution_benchmark.md').read()))
display(Image(f'{WORK}/results_v2/interpretation/attribution_example.png', width=900))

## 10. Pack the results for Claude

Creates **`My Drive/TFBS/TFBS_results_v2.zip`** with the tables, figures, trained models and JASPAR file. Download it from Drive and send it back in the chat. (The per-fold checkpoints are left out to keep it small.)

In [ ]:
import shutil, os
pack = '/content/TFBS_results'
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(f'{WORK}/results_v2', pack, ignore=shutil.ignore_patterns('checkpoints', 'quick_test'))
shutil.copy(LOCAL, pack)
shutil.make_archive(f'{WORK}/TFBS_results_v2', 'zip', pack)
print('Done:', f'{WORK}/TFBS_results_v2.zip', round(os.path.getsize(f'{WORK}/TFBS_results_v2.zip') / 1e6, 1), 'MB')

## 11. Check the website's backend with the new model (2 minutes)

Starts the real API here in Colab (not on your laptop) with the freshly trained model and sends it two real test sequences and one chatbot question. Bugs show up here instead of on your laptop. If this cell shows an error, send Claude a screenshot along with the results zip.

In [ ]:
!pip -q install fastapi uvicorn httpx
import shutil, sys, json, importlib
shutil.copytree(f'{WORK}/results_v2/final_models', f'{PROJECT}/ml/models/trained', dirs_exist_ok=True)
sys.path.insert(0, f'{PROJECT}/ml')
for m in ('api',):
    sys.modules.pop(m, None)
import api
from fastapi.testclient import TestClient
client = TestClient(api.app)
print(client.get('/').json())
test = pd.read_csv(f'{PROJECT}/ml/data/processed/test_clean.csv')
check = {}
for label in (1, 0):
    seq = test[test.label == label].sequence.iloc[3]
    r = client.post('/predict', json={'sequence': seq})
    assert r.status_code == 200, r.text
    d = r.json()
    check[label] = d['binding_probability']
    print(f"label {label}: p={d['binding_probability']} interval={d['confidence_interval']} -> {d['prediction_label']} | "
          f"motif: {d['visualizations']['sequence_logo']['matched_jaspar_id']} | found: {d['motifs_detected'][:5]}")
r = client.post('/chat', json={'question': 'Why is position 40 important?', 'mode': 'beginner',
                               'prediction': {'binding_probability': check[1], 'sequence': seq}})
print('chat:', r.json()['answer'][:200])
print('Backend check passed.')